1 Import thư viện
   ---------------

In [ ]:
pip install ipykernel pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

2 Import Dataset
---------------

In [ ]:
INPUT_PATH = Path("../data/chotot_final.csv")

OUTPUT_PATH = Path("../data/preproces_data.csv")
df = pd.read_csv(INPUT_PATH)
df.head()      # xem 5 dòng đầu


,title,area,Diện tích đất:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,...,Hướng ban công:,Đặc điểm căn hộ:,Loại hình văn phòng:,street,ward,district,city,Phân khu/Lô/Block/Tháp,has_floor_number,Price (trieu VND)
0,Tôi bán nhà xây mới 100% 4 lầu tặng nội thất 1%MG,36.3,36.3,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,4.7,7.5,3.0,...,Unknown,Unknown,Unknown,đường lê quang định,phường 11,quận bình thạnh,tp hồ chí minh,Không thuộc project/block,0,5500.0
1,BÁN NỀN KHU DÂN CƯ MINH THẮNG,115.0,115.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,19.0,3.0,...,Unknown,Unknown,Unknown,khu dân cư minh thắng - cà mau,phường 9,thành phố cà mau,cà mau,Không thuộc project/block,0,1570.0
2,Đất nền giá đầu tư becamex,250.0,250.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,10.0,25.0,3.0,...,Unknown,Unknown,Unknown,ql13,thị trấn lai uyên,huyện bàu bàng,bình dương,Không thuộc project/block,0,580.0
3,Cần bán 3 xào trồng tiêu đang thu hoạch,3031.0,3031.0,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,51.0,55.0,3.0,...,Unknown,Unknown,Unknown,đường 765,xã sông ray,huyện cẩm mỹ,đồng nai,Không thuộc project/block,0,1000.0
4,"Bán đất 440m2 (6 x 72) mặt tiền Tỉnh lộ 934, ST",440.0,440.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,72.0,3.0,...,Unknown,Unknown,Unknown,đường tỉnh lộ 934,xã viên bình,huyện trần đề,sóc trăng,Không thuộc project/block,0,580.0


Làm sạch column

In [ ]:
df.columns = df.columns.str.strip()

In ra các tất cả cột

In [ ]:
print(df.columns.tolist())

['title', 'area', 'Diện tích đất:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:', 'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:', 'Loại hình căn hộ:', 'Tổng số tầng:', 'Tầng số:', 'Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'street', 'ward', 'district', 'city', 'Phân khu/Lô/Block/Tháp', 'has_floor_number', 'Price (trieu VND)']


Drop đi title và area
- title vì dữ liệu quá nhiễu
- area thì trung với diện tích đất, và bị missing

In [ ]:
df = df.drop(columns=["title", "area"])
df.head() 


,Diện tích đất:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,...,Hướng ban công:,Đặc điểm căn hộ:,Loại hình văn phòng:,street,ward,district,city,Phân khu/Lô/Block/Tháp,has_floor_number,Price (trieu VND)
0,36.3,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,4.7,7.5,3.0,4.0,"Nhà ngõ, hẻm",...,Unknown,Unknown,Unknown,đường lê quang định,phường 11,quận bình thạnh,tp hồ chí minh,Không thuộc project/block,0,5500.0
1,115.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,19.0,3.0,4.0,"Nhà ngõ, hẻm",...,Unknown,Unknown,Unknown,khu dân cư minh thắng - cà mau,phường 9,thành phố cà mau,cà mau,Không thuộc project/block,0,1570.0
2,250.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,10.0,25.0,3.0,4.0,"Nhà ngõ, hẻm",...,Unknown,Unknown,Unknown,ql13,thị trấn lai uyên,huyện bàu bàng,bình dương,Không thuộc project/block,0,580.0
3,3031.0,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,51.0,55.0,3.0,4.0,"Nhà ngõ, hẻm",...,Unknown,Unknown,Unknown,đường 765,xã sông ray,huyện cẩm mỹ,đồng nai,Không thuộc project/block,0,1000.0
4,440.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,72.0,3.0,4.0,"Nhà ngõ, hẻm",...,Unknown,Unknown,Unknown,đường tỉnh lộ 934,xã viên bình,huyện trần đề,sóc trăng,Không thuộc project/block,0,580.0


In lại tất cả các column sau khi drop title và area

In [ ]:
print(df.columns.tolist())

['Diện tích đất:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:', 'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:', 'Loại hình căn hộ:', 'Tổng số tầng:', 'Tầng số:', 'Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'street', 'ward', 'district', 'city', 'Phân khu/Lô/Block/Tháp', 'has_floor_number', 'Price (trieu VND)']


3 Kiểm tra lần nữa xem có freature nào bị missing không
-------------------------------------------------------

In [ ]:
missing_df = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

missing_df = missing_df.sort_values(by="missing_percent", ascending=False)

missing_df

,missing_count,missing_percent
Diện tích đất:,0,0.0
Hướng cửa chính:,0,0.0
Giấy tờ pháp lý:,0,0.0
Đặc điểm nhà/đất:,0,0.0
Loại hình đất:,0,0.0
Chiều ngang:,0,0.0
Chiều dài:,0,0.0
Số phòng ngủ:,0,0.0
Số phòng vệ sinh:,0,0.0
Loại hình nhà ở:,0,0.0


Nhận xét: Không còn bị missing

4 Kiểm tra giá trị Unique của các cột street, ward and district
--------------------------------------------------------------

In [ ]:
for col in ["street", "ward", "district"]:
    print(col, ":", df[col].nunique())

street : 2865
ward : 986
district : 255


In [ ]:
for col in ["street", "ward", "district"]:
    ratio = df[col].nunique() / len(df)
    print(f"{col}: {ratio:.2%}")

street: 41.11%
ward: 14.15%
district: 3.66%


Unique values của street và ward quá nhiều. Nếu cố đưa vào model sẽ bị overfit.
TH1: drop street và ward column
TH2: lấy top K và các giá trị con lại gán other

5 Kiểm tra các giá trị Unknown
-------------------------------

In [ ]:
unknown_df = pd.DataFrame({
    "unknown_count": (df == "Unknown").sum(),
    "unknown_percent": ((df == "Unknown").sum() / len(df)) * 100
})

unknown_df = unknown_df.sort_values(by="unknown_percent", ascending=False)

unknown_df

,unknown_count,unknown_percent
Loại hình văn phòng:,255,3.659062
Đặc điểm căn hộ:,57,0.817908
ward,36,0.516573
Hướng ban công:,35,0.502224
city,28,0.401779
district,28,0.401779
street,23,0.330033
Đặc điểm nhà/đất:,0,0.000000
Diện tích đất:,0,0.000000
Hướng cửa chính:,0,0.000000


## 6 Drop street and ward column

Sau khi họp lại thì cả nhóm quyết định drop các feature street và ward.
Lý do: 
- Các giá trị của 2 feature này quá nhiều. Nếu cố đưa vào model sẽ dễ kiến model bị overfit
- Thực tế thì giá nhà sẽ bị ảnh hưởng phần lớn bởi Quận, Huyện nên ta có thể bỏ 2 model này.
- column ward không ảnh hưởng quá nhiều tới giá nhà

In [ ]:
df = df.drop(columns=["street", "ward"])

df.head()

,Diện tích đất:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,...,Tổng số tầng:,Tầng số:,Hướng ban công:,Đặc điểm căn hộ:,Loại hình văn phòng:,district,city,Phân khu/Lô/Block/Tháp,has_floor_number,Price (trieu VND)
0,36.3,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,4.7,7.5,3.0,4.0,"Nhà ngõ, hẻm",...,4.0,-1.0,Unknown,Unknown,Unknown,quận bình thạnh,tp hồ chí minh,Không thuộc project/block,0,5500.0
1,115.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,19.0,3.0,4.0,"Nhà ngõ, hẻm",...,4.0,-1.0,Unknown,Unknown,Unknown,thành phố cà mau,cà mau,Không thuộc project/block,0,1570.0
2,250.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,10.0,25.0,3.0,4.0,"Nhà ngõ, hẻm",...,4.0,-1.0,Unknown,Unknown,Unknown,huyện bàu bàng,bình dương,Không thuộc project/block,0,580.0
3,3031.0,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,51.0,55.0,3.0,4.0,"Nhà ngõ, hẻm",...,4.0,-1.0,Unknown,Unknown,Unknown,huyện cẩm mỹ,đồng nai,Không thuộc project/block,0,1000.0
4,440.0,Đông Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,72.0,3.0,4.0,"Nhà ngõ, hẻm",...,4.0,-1.0,Unknown,Unknown,Unknown,huyện trần đề,sóc trăng,Không thuộc project/block,0,580.0


## 7 Drop các sample có giá trị Unknown

Các giá trị Unknow tiền thân là các giá trị missing NaN. 
Lý do drop:
- Các giá trịnh Unknown được hiển thị vào các feature là các giá trị định tính chứ không phải định lượng.
- Bên cạnh đó thì các sample có unknown có drop cũng sẽ ít ảnh hưởng đến chất lượng model.

In [ ]:
# Xóa tất cả dòng có chứa "Unknown"
df = df[~df.isin(["Unknown"]).any(axis=1)]

# Reset index
df = df.reset_index(drop=True)

print(df.shape)

(6692, 25)


## 8. Xử lý các outlier được phát hiện từ phần EDA

In [ ]:
# 1. Hard-cap các giá trị vật lý vô lý (domain knowledge)
# Chiều ngang / chiều dài nhà dân dụng thực tế <= 100m
if "Chiều ngang:" in df.columns:
    df = df[df["Chiều ngang:"] <= 100]

if "Chiều dài:" in df.columns:
    df = df[df["Chiều dài:"] <= 100]

# Diện tích <= 500 m² (nhà ở, không phải đất nông nghiệp)
if "Diện tích đất:" in df.columns:
    df = df[df["Diện tích đất:"] <= 500]

# Số tầng, phòng ngủ, phòng vệ sinh hợp lý
if "Tổng số tầng:" in df.columns:
    df = df[df["Tổng số tầng:"] <= 40]

if "Số phòng ngủ:" in df.columns:
    df = df[df["Số phòng ngủ:"] <= 15]

if "Số phòng vệ sinh:" in df.columns:
    df = df[df["Số phòng vệ sinh:"] <= 15]

print("Shape sau hard-cap domain:", df.shape)

In [ ]:
# 2. IQR Clipping cho Price và các cột diện tích
# Dùng clipping thay vì drop để giữ nhiều dữ liệu hơn
# (phù hợp hơn với RF, XGBoost vì chúng ít nhạy cảm với outlier)
TARGET = "Price (trieu VND)"
iqr_clip_cols = [TARGET]

if "Diện tích đất:" in df.columns:
    iqr_clip_cols.append("Diện tích đất:")

for col in iqr_clip_cols:
    if col not in df.columns:
        continue
    q1 = df[col].quantile(0.01)   # cắt 1% dưới
    q3 = df[col].quantile(0.99)   # cắt 1% trên
    before = len(df)
    df = df[(df[col] >= q1) & (df[col] <= q3)]
    print(f"  [{col}] drop {before - len(df)} dòng ngoài [{q1:.1f}, {q3:.1f}]")

print("Shape sau IQR filter:", df.shape)

In [ ]:
# 3. Log-transform Price (giảm skewness cho Linear Regression)
# Tạo cột log_price để dùng khi train Linear Regression
# RF và XGBoost có thể dùng price gốc, nhưng log thường vẫn tốt hơn
df["log_price"] = np.log1p(df[TARGET])

print("\nSkewness Price gốc :", round(df[TARGET].apply(np.log1p).skew(), 4), "(sau log)")
print("Skewness Price gốc :", round(df[TARGET].skew(), 4), "(trước log)")

In [ ]:
# 4. Kiểm tra nhanh outlier còn lại
def iqr_outlier_pct(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    mask = (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)
    return round(mask.sum() / len(series) * 100, 2)

numeric_cols = df.select_dtypes(include="number").columns.tolist()
outlier_pct = {c: iqr_outlier_pct(df[c]) for c in numeric_cols}
print("\nOutlier % còn lại sau xử lý:")
for c, v in sorted(outlier_pct.items(), key=lambda x: -x[1]):
    print(f"  {c}: {v}%")

df = df.reset_index(drop=True)
print("\nShape cuối:", df.shape)

## 8 Lưu lại file csv đã preprocessing

In [ ]:
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Đã lưu file tại: {OUTPUT_PATH}")

Đã lưu file tại: ..\data\preproces_data.csv


## 9 Kiểm tra lại lần cuối

In [ ]:
print(df.columns.tolist())

['Diện tích đất:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:', 'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:', 'Loại hình căn hộ:', 'Tổng số tầng:', 'Tầng số:', 'Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'district', 'city', 'Phân khu/Lô/Block/Tháp', 'has_floor_number', 'Price (trieu VND)']


In [ ]:
unknown_df = pd.DataFrame({
    "unknown_count": (df == "unknown").sum(),
    "unknown_percent": ((df == "unknown").sum() / len(df)) * 100
})

unknown_df = unknown_df.sort_values(
    by="unknown_percent",
    ascending=False
)

unknown_df

,unknown_count,unknown_percent
Diện tích đất:,0,0.0
Hướng cửa chính:,0,0.0
Giấy tờ pháp lý:,0,0.0
Đặc điểm nhà/đất:,0,0.0
Loại hình đất:,0,0.0
Chiều ngang:,0,0.0
Chiều dài:,0,0.0
Số phòng ngủ:,0,0.0
Số phòng vệ sinh:,0,0.0
Loại hình nhà ở:,0,0.0
